# 09 — GRPO using TRL

## Learning objectives

- Verify exact-answer rewards, parser edge cases and metadata.
- Specify grouping, reference KL and loss variant explicitly.
- Run local tiny trainer smoke and optional pretrained arithmetic fine-tuning.

This notebook runs a small CPU demonstration by default. Synthetic experiments are educational, not evidence of LLM alignment. Pretrained-model downloads and GPU training require explicit opt-in. All runtime outputs stay inside `RLHF_NOTEBOOKS/`.

**Source connection:** GRPO source group examples and notebook batch1/G4, lexical correctness and `.isdigit()` pitfalls; notes Chapter06; exact TRL0.22.2 config/source.

[Learning order and setup](README.md).

In [1]:
from pathlib import Path
import os, sys, json, math, random, copy, re, subprocess, hashlib
# Find this collection when launched from the repository, module, or notebook directory.
candidates = [Path.cwd(), Path.cwd()/'RLHF_NOTEBOOKS', Path.cwd()/'13_rlhf_ppo_dpo_orpo/RLHF_NOTEBOOKS']
ROOT = next((p.resolve() for p in candidates if (p/'source_inspection.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter from RLHF_NOTEBOOKS, its parent, or the repository root.')
RUNTIME = ROOT/'runtime'; RUNTIME.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(RUNTIME/'matplotlib')
os.environ['HF_HOME'] = str(RUNTIME/'huggingface')
os.environ['HF_DATASETS_CACHE'] = str(RUNTIME/'huggingface/datasets')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
random.seed(42); np.random.seed(42); torch.manual_seed(42)
torch.set_num_threads(1)
print('Python', sys.version.split()[0], 'PyTorch', torch.__version__, 'CUDA', torch.cuda.is_available())

Python 3.13.9 PyTorch 2.8.0+cpu CUDA False


## Reward and group design
Group size $G=4$ must divide the effective generation batch. The CPU recipe uses batch4/accumulation1 on one process. In multi-process training check global generation settings. Set `loss_type='grpo'` explicitly because this release defaults to DAPO. `beta=.01` is reference KL coefficient, not DPO scaling. `scale_rewards='group'` uses within-prompt normalization, with library std conventions differing from notebook08's population std.

Correctness parses one numeric answer with decimal normalization, so `2.0` can equal `2` and negative answers are not rejected by `.isdigit()`. Extra text fails by design; format validity and task correctness remain distinct. No lexical similarity or uncertainty-penalty reward is used. A random model may produce all-wrong groups: zero variance means no correctness gradient, though KL still applies.

In [2]:
def read_preferences():
    path = ROOT.parent/'reward_preferences_dataset.jsonl'
    if path.exists():
        raw = path.read_bytes(); origin = str(path)
    else:
        # Originals are currently missing from the working tree. Read their Git blob;
        # never restore, overwrite or silently substitute invented preference records.
        gitroot = Path(subprocess.check_output(['git','rev-parse','--show-toplevel'], cwd=ROOT, text=True).strip())
        rel = (ROOT.parent/'reward_preferences_dataset.jsonl').relative_to(gitroot).as_posix()
        raw = subprocess.check_output(['git','show','HEAD:'+rel], cwd=gitroot)
        origin = 'HEAD:'+rel
    expected = json.loads((ROOT/'source_inspection.json').read_text())
    digest = next(r['sha256'] for r in expected if r['source']=='reward_preferences_dataset.jsonl')
    assert hashlib.sha256(raw).hexdigest() == digest, 'Dataset changed: re-audit before proceeding'
    rows = [json.loads(line) for line in raw.decode('utf-8').splitlines()]
    for row in rows:
        assert set(row)=={'prompt','chosen','rejected'}
        assert all(isinstance(row[k],str) and row[k].strip() for k in row)
        assert row['chosen'].strip()!=row['rejected'].strip()
    assert len({r['prompt'].strip().lower() for r in rows})==len(rows)
    print('Dataset origin:',origin,'records:',len(rows),'fields:',list(rows[0]))
    return rows
rows = read_preferences()
indices = np.random.default_rng(42).permutation(len(rows))
train_rows = [rows[i] for i in indices[:14]]
valid_rows = [rows[i] for i in indices[14:17]]
test_rows = [rows[i] for i in indices[17:]]
assert not ({r['prompt'] for r in train_rows} & {r['prompt'] for r in valid_rows+test_rows})
print('Prompt-separated split:',len(train_rows),len(valid_rows),len(test_rows))

Dataset origin: HEAD:13_rlhf_ppo_dpo_orpo/reward_preferences_dataset.jsonl records: 20 fields: ['prompt', 'chosen', 'rejected']
Prompt-separated split: 14 3 3


In [3]:
# Local tiny GPT-2 architecture and WordLevel tokenizer: NO pretrained downloads.
from importlib import metadata
import inspect
from tokenizers import Tokenizer,models,pre_tokenizers
from transformers import PreTrainedTokenizerFast,GPT2Config,GPT2LMHeadModel,GPT2ForSequenceClassification
from datasets import Dataset
from trl import DPOTrainer,DPOConfig,GRPOTrainer,GRPOConfig,ORPOTrainer,ORPOConfig,RewardTrainer,RewardConfig,PPOTrainer,PPOConfig
VERSIONS={name:metadata.version(name) for name in ('torch','transformers','trl','datasets','peft','accelerate')}
print(VERSIONS)
assert metadata.version('trl')=='0.22.2', 'Use the pinned environment; APIs change across TRL releases.'
for cls in [DPOTrainer,GRPOTrainer,ORPOTrainer,PPOTrainer]: print(cls.__name__,inspect.signature(cls.__init__))
texts=['Question: '+r['prompt']+' Answer: '+r[s] for r in rows for s in ('chosen','rejected')]
words=sorted(set(re.findall(r'\w+|[^\w\s]', ' '.join(texts)+' What is 1 + 1 ? 2 3 4 - .')))
word_vocab={'<pad>':0,'<eos>':1,'<unk>':2,'<bos>':3,**{w:i+4 for i,w in enumerate(words)}}
backend=Tokenizer(models.WordLevel(word_vocab,unk_token='<unk>'));backend.pre_tokenizer=pre_tokenizers.Whitespace()
tokenizer=PreTrainedTokenizerFast(tokenizer_object=backend,pad_token='<pad>',eos_token='<eos>',unk_token='<unk>',bos_token='<bos>')
tokenizer.chat_template="{% for message in messages %}{{ message['role'] + ': ' + message['content'] + '\\n' }}{% endfor %}{% if add_generation_prompt %}{{ 'assistant: ' }}{% endif %}"
def lm_config():
    return GPT2Config(vocab_size=len(tokenizer),n_positions=256,n_ctx=256,n_embd=32,n_layer=1,n_head=2,pad_token_id=0,eos_token_id=1,bos_token_id=3,resid_pdrop=0.,embd_pdrop=0.,attn_pdrop=0.)
def hf_pairs(examples):
    return Dataset.from_list([{'prompt':'Question: '+r['prompt']+' Answer:','chosen':' '+r['chosen']+tokenizer.eos_token,'rejected':' '+r['rejected']+tokenizer.eos_token} for r in examples])
RUN_TRL_SMOKE = True   # two steps on randomly initialized tiny LOCAL model, CPU-friendly
RUN_PRETRAINED_GPU = False  # opt-in; downloads a real checkpoint only when True
print('Tokenizer distinct PAD/EOS:',tokenizer.pad_token_id,tokenizer.eos_token_id)
assert tokenizer.pad_token_id != tokenizer.eos_token_id

/home/akhand/Desktop/LLM FINETUNING/SLM-Lab-Small-Language-Model-Experiments/13_rlhf_ppo_dpo_orpo/RLHF_NOTEBOOKS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


{'torch': '2.8.0+cpu', 'transformers': '4.56.2', 'trl': '0.22.2', 'datasets': '4.3.0', 'peft': '0.17.1', 'accelerate': '1.10.1'}
DPOTrainer (self, model: Union[str, torch.nn.modules.module.Module, transformers.modeling_utils.PreTrainedModel], ref_model: Union[transformers.modeling_utils.PreTrainedModel, torch.nn.modules.module.Module, str, NoneType] = None, args: Optional[trl.trainer.dpo_config.DPOConfig] = None, data_collator: Optional[transformers.data.data_collator.DataCollator] = None, train_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, NoneType] = None, eval_dataset: Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset, dict[str, Union[datasets.arrow_dataset.Dataset, datasets.iterable_dataset.IterableDataset]], NoneType] = None, processing_class: Union[transformers.tokenization_utils_base.PreTrainedTokenizerBase, transformers.image_processing_utils.BaseImageProcessor, transformers.feature_extraction_utils.Featu

In [4]:
def integer_answer(text):
    # One final number only; supports signed integers/decimals, rejects junk/extra answers.
    from decimal import Decimal,InvalidOperation
    text=text.strip()
    if not re.fullmatch(r'[+-]?\d+(?:\.\d+)?',text):return None
    try:return Decimal(text)
    except InvalidOperation:return None

def exact_reward(completions,answer,**kwargs):
    result=[]
    for completion,gold in zip(completions,answer):
        text=completion if isinstance(completion,str) else completion[-1]['content']
        parsed=integer_answer(text);result.append(float(parsed is not None and parsed==integer_answer(str(gold))))
    return result
assert exact_reward(['-2','2.0','2 dogs',''],['-2','2','2','0'])==[1.,1.,0.,0.]
# Synthetic arithmetic task. answer is created metadata, NOT a field of local JSONL.
arithmetic=Dataset.from_list([{'prompt':f'What is {a} + {b} ? Answer only the number:', 'answer':str(a+b)} for a,b in [(1,1),(1,2),(2,2),(2,1)]])
if RUN_TRL_SMOKE:
    tokenizer.padding_side='left';model=GPT2LMHeadModel(lm_config())
    # TRL builds non-PEFT reference from config._name_or_path: save/reload local
    # random initialization so the reference resolves locally rather than the Hub.
    initial_path=RUNTIME/'grpo_initial';model.save_pretrained(str(initial_path))
    model=GPT2LMHeadModel.from_pretrained(str(initial_path),local_files_only=True)
    args=GRPOConfig(output_dir=str(RUNTIME/'grpo_cpu'),max_steps=2,per_device_train_batch_size=4,per_device_eval_batch_size=4,
                    gradient_accumulation_steps=1,num_generations=4,max_prompt_length=64,max_completion_length=4,
                    loss_type='grpo',scale_rewards='group',beta=.01,num_iterations=1,epsilon=.2,temperature=1.,
                    learning_rate=1e-4,use_cpu=True,bf16=False,fp16=False,report_to='none',save_strategy='steps',save_steps=1,logging_steps=1,
                    remove_unused_columns=False,use_vllm=False,seed=42)
    trainer=GRPOTrainer(model=model,args=args,processing_class=tokenizer,reward_funcs=exact_reward,train_dataset=arithmetic)
    result=trainer.train();trainer.save_model(str(RUNTIME/'grpo_cpu/final'));tokenizer.save_pretrained(str(RUNTIME/'grpo_cpu/final'))
    print('Actual random-model GRPO smoke:',result)
    print('Most groups may have zero correctness variance; this is API verification, not learning evidence.')

Step,Training Loss
1,0.000000
2,0.000000


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Actual random-model GRPO smoke: TrainOutput(global_step=2, training_loss=0.0, metrics={'train_runtime': 0.0522, 'train_samples_per_second': 153.355, 'train_steps_per_second': 38.339, 'total_flos': 0.0, 'train_loss': 0.0})
Most groups may have zero correctness variance; this is API verification, not learning evidence.


## Optional reasoning workflow

The real-model branch is off. It uses synthetic arithmetic with separate validation, structured prompts, no vLLM dependency, explicit grouping and short generation. Log reward mean/std, constant-group fraction, numeric parse success, lengths and KL. Increase problem difficulty gradually; do not add random reward noise to disguise zero variance.

In [5]:
if RUN_PRETRAINED_GPU:
    assert torch.cuda.is_available()
    from transformers import AutoTokenizer,AutoModelForCausalLM
    from peft import LoraConfig
    MODEL_ID='HuggingFaceTB/SmolLM2-135M-Instruct'
    tok=AutoTokenizer.from_pretrained(MODEL_ID);tok.padding_side='left'
    if tok.pad_token_id is None:tok.pad_token=tok.eos_token
    examples=[]
    for a in range(1,6):
        for b in range(1,6):
            examples.append({'prompt':[{'role':'user','content':f'What is {a}+{b}? Return just the number.'}],'answer':str(a+b)})
    # Explicit split; these synthetic arithmetic records are not human preferences.
    training=Dataset.from_list(examples[:20]);validation=Dataset.from_list(examples[20:])
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=dtype);model.config.use_cache=False
    config=GRPOConfig(output_dir=str(RUNTIME/'grpo_gpu'),max_steps=5,per_device_train_batch_size=4,per_device_eval_batch_size=4,
                      num_generations=4,gradient_accumulation_steps=1,max_prompt_length=128,max_completion_length=32,
                      loss_type='grpo',scale_rewards='group',beta=.01,num_iterations=1,epsilon=.2,temperature=.9,
                      learning_rate=1e-5,bf16=dtype==torch.bfloat16,fp16=False,use_vllm=False,remove_unused_columns=False,
                      eval_strategy='steps',eval_steps=5,save_strategy='steps',save_steps=5,report_to='none')
    trainer=GRPOTrainer(model=model,args=config,processing_class=tok,reward_funcs=exact_reward,train_dataset=training,eval_dataset=validation,
                        peft_config=LoraConfig(task_type='CAUSAL_LM',r=8,lora_alpha=16,target_modules=['q_proj','v_proj']))
    trainer.train();print(trainer.evaluate());trainer.save_model(str(RUNTIME/'grpo_gpu/final'));tok.save_pretrained(str(RUNTIME/'grpo_gpu/final'))

## Exercises

1. Test parser equivalence, malformed tags and empty completions.
2. Intentionally misconfigure G and observe config validation.
3. Compare loss_type grpo/dapo only after stating the different weighting.

## Key takeaways

Recompute the worked examples, inspect masks and gradients, and evaluate independently of the training objective. A small training-loss decrease does not establish generalization or alignment. Record the model, data split, seeds and dependency versions before interpreting results.